In [ ]:
import sys
sys.path.append('../..')

import pandas as pd
import re
from eeg.splitter import split_report, split_routine, split_ltm, split_emu

EEG_CSV = '../../data/eeg_reports.csv'
KEEP_SERVICES = ['Routine', 'LTM', 'EMU']

df = pd.read_csv(EEG_CSV, engine='python', on_bad_lines='skip')
df = df[df['ServiceName'].isin(KEEP_SERVICES)].copy()
print(f'Loaded: {len(df):,} reports')
print(df['ServiceName'].value_counts())

In [ ]:
for svc in KEEP_SERVICES:
    row = df[df['ServiceName'] == svc].iloc[0]
    result = split_report(row)
    print(f'{"="*60}')
    print(f'SERVICE: {svc}')
    print(f'{"="*60}')
    print(f'INPUT length:  {len(row["ReportTextRedacted"])} chars')
    print(f'OUTPUT length: {len(result) if result else 0} chars')
    print()
    print(result if result else 'NONE -- extraction failed')
    print()

In [ ]:
df['finding_text'] = df.apply(split_report, axis=1)

print('Extraction coverage by service type:')
for svc in KEEP_SERVICES:
    subset = df[df['ServiceName'] == svc]
    valid = subset['finding_text'].notna().sum()
    total = len(subset)
    print(f'  {svc}: {valid:,} / {total:,} ({100*valid/total:.1f}%)')

print()
total_valid = df['finding_text'].notna().sum()
print(f'Overall: {total_valid:,} / {len(df):,} ({100*total_valid/len(df):.1f}%)')

In [ ]:
import matplotlib.pyplot as plt

df['output_len'] = df['finding_text'].str.len()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, svc in zip(axes, KEEP_SERVICES):
    subset = df[df['ServiceName'] == svc]['output_len'].dropna()
    ax.hist(subset, bins=50, edgecolor='none')
    ax.set_title(svc)
    ax.set_xlabel('characters')
    ax.set_ylabel('reports')
    ax.axvline(subset.median(), color='red', linestyle='--',
                label=f'median {subset.median():.0f}')
    ax.legend(fontsize=8)
plt.suptitle('Extracted finding length by service type', y=1.02)
plt.tight_layout()
plt.show()

print(df.groupby('ServiceName')['output_len'].describe().round(0))

In [ ]:
failures = df[df['finding_text'].isna()]
print(f'Total failures: {len(failures):,}')
print(failures['ServiceName'].value_counts())
print()

# print first 3 failure cases per type to understand why they failed
for svc in KEEP_SERVICES:
    subset = failures[failures['ServiceName'] == svc]
    if len(subset) == 0:
        print(f'{svc}: no failures')
        continue
    print(f'=== {svc} failures (showing first 3) ===')
    for _, row in subset.head(3).iterrows():
        print(f'--- report length: {len(row["ReportTextRedacted"])} chars ---')
        print(row['ReportTextRedacted'][:500])
        print()

In [ ]:
SHORT_THRESHOLD = 100  # chars

short = df[df['output_len'] < SHORT_THRESHOLD]
print(f'Outputs under {SHORT_THRESHOLD} chars: {len(short):,}')
print(short['ServiceName'].value_counts())
print()

for svc in KEEP_SERVICES:
    subset = short[short['ServiceName'] == svc]
    if len(subset) == 0:
        continue
    print(f'=== {svc} short outputs (first 3) ===')
    for _, row in subset.head(3).iterrows():
        print(f'OUTPUT: {repr(row["finding_text"])}')
        print(f'INPUT snippet: {row["ReportTextRedacted"][:300]}')
        print()

In [ ]:
for svc in KEEP_SERVICES:
    subset = df[
        (df['ServiceName'] == svc) &
        df['finding_text'].notna()
    ].sample(5, random_state=42)
    print(f'{"="*60}')
    print(f'SERVICE: {svc} -- 5 random outputs')
    print(f'{"="*60}')
    for _, row in subset.iterrows():
        print(f'--- {len(row["finding_text"])} chars ---')
        print(row['finding_text'][:400])
        print()

In [ ]:
# Add known tricky reports here as you find them during testing.
# Format: (service_name, report_text, expected_substring_in_output)

REGRESSION_CASES = [
    # example -- add real cases as you find them:
    # ('Routine', 'This is an abnormal EEG...', 'abnormal'),
]

if not REGRESSION_CASES:
    print('No regression cases yet -- add them as you find failures in cells 5 and 6')
else:
    passed = 0
    for i, (svc, text, expected) in enumerate(REGRESSION_CASES):
        row = pd.Series({'ServiceName': svc, 'ReportTextRedacted': text})
        result = split_report(row)
        ok = result is not None and expected in result
        status = 'PASS' if ok else 'FAIL'
        print(f'[{status}] Case {i+1} ({svc})')
        if not ok:
            print(f'  expected: {repr(expected)}')
            print(f'  got: {repr(result[:200] if result else None)}')
        passed += ok
    print(f'\n{passed}/{len(REGRESSION_CASES)} passed')